<a href="https://colab.research.google.com/github/visionbyangelic/Brain-Aging/blob/main/data/Feature_Compatibility_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Compatibility Assessment

This notebook evaluates the compatibility of the MRI-derived feature spaces established for the OpenBHB and OASIS-3 cohorts in Build 1.

The purpose of this stage is to determine whether the available measurements can support a valid cross-dataset brain-age modelling pipeline. Because the normative model will be developed using healthy OpenBHB participants and subsequently evaluated on OASIS-3, the predictors must represent comparable biological measurements across datasets.

The assessment will:

1. Compare the available MRI-derived features in OpenBHB and OASIS-3.
2. Identify features that are shared or potentially equivalent between datasets.
3. Examine feature definitions, units, and scaling where information is available.
4. Determine whether the current data support a reduced shared-feature approach or require regional feature parity.
5. Document the selected feature-compatibility strategy for Build 1.

Two routes are considered:

- **Option A — Reduced shared features:** use only MRI measures that are demonstrably comparable across both datasets.
- **Option B — Regional parity:** obtain or construct a compatible regional morphometric feature space across both datasets.

No brain-age model is trained in this notebook. The purpose is to establish the feature space and compatibility decision required before model development.

In [2]:
# ============================================================
# Feature Compatibility Assessment
# Environment and Paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/ANR_BrainAge")

# ------------------------------------------------------------
# Manifest directories
# ------------------------------------------------------------
OASIS_MAN = ROOT / "manifests" / "oasis3"
OPENBHB_MAN = ROOT / "manifests" / "openbhb"

# ------------------------------------------------------------
# Feature inventories generated in the previous stages
# ------------------------------------------------------------
OASIS_INVENTORY = OASIS_MAN / "OASIS3_feature_inventory.csv"
OPENBHB_INVENTORY = OPENBHB_MAN / "OpenBHB_feature_inventory.csv"

print("Environment initialized.")
print(f"Project root:          {ROOT}")
print(f"OASIS-3 inventory:     {OASIS_INVENTORY}")
print(f"OpenBHB inventory:     {OPENBHB_INVENTORY}")

print("\nFile checks:")
print(f"OASIS-3 inventory exists:  {OASIS_INVENTORY.exists()}")
print(f"OpenBHB inventory exists:  {OPENBHB_INVENTORY.exists()}")

Environment initialized.
Project root:          /content/drive/MyDrive/ANR_BrainAge
OASIS-3 inventory:     /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3/OASIS3_feature_inventory.csv
OpenBHB inventory:     /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/OpenBHB_feature_inventory.csv

File checks:
OASIS-3 inventory exists:  True
OpenBHB inventory exists:  True


In [3]:
# ============================================================
# Load OASIS-3 and OpenBHB Feature Inventories
# ============================================================

oasis_inventory = pd.read_csv(OASIS_INVENTORY)
openbhb_inventory = pd.read_csv(OPENBHB_INVENTORY)

print("Feature inventories loaded.")
print(f"OASIS-3 features:  {len(oasis_inventory)}")
print(f"OpenBHB features:  {len(openbhb_inventory)}")

print("\nOASIS-3 inventory:")
print(oasis_inventory.to_string(index=False))

print("\nOpenBHB inventory:")
print(openbhb_inventory.to_string(index=False))

Feature inventories loaded.
OASIS-3 features:  196
OpenBHB features:  4

OASIS-3 inventory:
                              feature               type
                      IntraCranialVol             Volume
                          lhCortexVol             Volume
                          rhCortexVol             Volume
                            CortexVol             Volume
                       SubCortGrayVol             Volume
                         TotalGrayVol             Volume
                    SupraTentorialVol             Volume
             lhCorticalWhiteMatterVol             Volume
             rhCorticalWhiteMatterVol             Volume
               CorticalWhiteMatterVol             Volume
                 3rd-Ventricle_volume             Volume
                 4th-Ventricle_volume             Volume
                 5th-Ventricle_volume             Volume
                    Brain-Stem_volume             Volume
                   CC_Anterior_volume             Vol

## Feature Space Comparison: OpenBHB and OASIS-3

The feature inventories established in the preceding stages reveal a substantial difference in the available MRI-derived feature spaces.

OASIS-3 currently provides **196 MRI-derived structural features**, including global and regional volumetric measures, cortical thickness measures, surface-area measures, and vertex counts. The feature inventory includes both hemispheric and regional morphometric measurements.

OpenBHB, in the current Build 1 manifest, provides **four MRI-derived summary measures**:

- `tiv` — total intracranial volume
- `csfv` — cerebrospinal fluid volume
- `gmv` — gray matter volume
- `wmv` — white matter volume

All four OpenBHB measures are numeric and contain no missing values.

### Candidate Cross-Dataset Correspondence

Based on the feature names alone, the following OASIS-3 measures appear to be potential counterparts to the OpenBHB summary measures:

| OpenBHB | OASIS-3 candidate |
|---|---|
| `tiv` | `IntraCranialVol` |
| `csfv` | `CSF_volume` |
| `gmv` | `TotalGrayVol` |
| `wmv` | `CorticalWhiteMatterVol` |

These are currently treated as **candidate correspondences only**. Similar naming does not establish that the measurements were derived using identical definitions, preprocessing procedures, units, or anatomical conventions.

### Purpose of the Compatibility Assessment

Before training the normative brain-age model, these candidate correspondences must therefore be verified using the relevant dataset documentation and feature definitions.

This verification is necessary because the Build 1 model is intended to learn a normative relationship between structural MRI measurements and chronological age in OpenBHB and subsequently be applied to OASIS-3. Using measurements that are not genuinely comparable across datasets could introduce systematic dataset or preprocessing differences that may be incorrectly interpreted as biological differences in brain aging.

The compatibility assessment will therefore determine whether the current datasets support:

- **Option A — Reduced shared features:** use only measurements demonstrated to be comparable across OpenBHB and OASIS-3; or
- **Option B — Regional feature parity:** obtain a compatible regional morphometric feature space before cross-dataset modelling.

No final feature set is selected at this stage, and no brain-age model is trained until this compatibility decision has been established.

---

In [4]:
# ============================================================
# Inspect OpenBHB Project Data and Manifest Files
# ============================================================

# OpenBHB data and manifest directories
OPENBHB_DATA = ROOT / "data" / "openbhb"
OPENBHB_MAN = ROOT / "manifests" / "openbhb"

print("OpenBHB directories")
print("=" * 60)
print(f"Data directory:      {OPENBHB_DATA}")
print(f"Data exists:         {OPENBHB_DATA.exists()}")
print(f"Manifest directory:  {OPENBHB_MAN}")
print(f"Manifest exists:     {OPENBHB_MAN.exists()}")

# ------------------------------------------------------------
# List files in the OpenBHB data directory
# ------------------------------------------------------------

if OPENBHB_DATA.exists():
    print("\nFiles/directories in OpenBHB data:")
    for item in sorted(OPENBHB_DATA.iterdir()):
        print(f"  {item.name}")

# ------------------------------------------------------------
# List files in the OpenBHB manifest directory
# ------------------------------------------------------------

if OPENBHB_MAN.exists():
    print("\nFiles in OpenBHB manifests:")
    for item in sorted(OPENBHB_MAN.iterdir()):
        print(f"  {item.name}")

OpenBHB directories
Data directory:      /content/drive/MyDrive/ANR_BrainAge/data/openbhb
Data exists:         False
Manifest directory:  /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb
Manifest exists:     True

Files in OpenBHB manifests:
  OpenBHB_feature_inventory.csv
  openBHB_3T_candidates.csv
  openBHB_3T_candidates_with_scanner_status.csv
  openBHB_3T_site_acquisition_summary.csv
  openBHB_approved_RELAXED_TimTrio.csv
  openBHB_approved_RELAXED_with_FreeSurfer.csv
  openBHB_approved_STRICT.csv
  openBHB_approved_manifest.csv
  openBHB_excluded_field_strength.csv
  openBHB_excluded_scanners.csv
  openBHB_master_manifest.csv
  openBHB_scanner_mapping_template.csv


In [5]:
# ============================================================
# Inspect OpenBHB FreeSurfer-Linked Manifest
# ============================================================

OPENBHB_FS_MANIFEST = (
    OPENBHB_MAN / "openBHB_approved_RELAXED_with_FreeSurfer.csv"
)

print("OpenBHB FreeSurfer-linked manifest")
print("=" * 60)
print(f"Path: {OPENBHB_FS_MANIFEST}")
print(f"File exists: {OPENBHB_FS_MANIFEST.exists()}")

if OPENBHB_FS_MANIFEST.exists():
    openbhb_fs = pd.read_csv(OPENBHB_FS_MANIFEST)

    print(f"\nShape: {openbhb_fs.shape}")
    print(f"Columns: {len(openbhb_fs.columns)}")

    print("\nColumns:")
    print(openbhb_fs.columns.tolist())

OpenBHB FreeSurfer-linked manifest
Path: /content/drive/MyDrive/ANR_BrainAge/manifests/openbhb/openBHB_approved_RELAXED_with_FreeSurfer.csv
File exists: True

Shape: (3240, 33)
Columns: 33

Columns:
['participant_id', 'study', 'sex', 'age', 'site', 'diagnosis', 'tiv', 'csfv', 'gmv', 'wmv', 'magnetic_field_strength', 'acquisition_setting', 'siteXacq', 'split', 'reconall-euler', 'cat12vbm-ncr', 'cat12vbm-iqr', 'quasiraw-corr', 'dataset', 'cohort', 'clinical_group', 'manufacturer_x', 'scanner_model_x', 'scanner_verified', 'scanner_eligible', 'field_strength_eligible', 'include_stage1', 'exclusion_reason', 'source_dataset', 'scanner_status', 'manufacturer_y', 'scanner_model_y', 'evidence_source']
